# Research Direction - Raw Data First

Research question: can five-minute bar data support an honest directional
classifier that beats simple lower-bound comparators under chronological
train/validation screening?

Scope: `validation_only`.

This notebook is the lightweight research-direction entry point. It does not
import the local project package or prior notebook modules.
It records the raw data source, the Colab opening workflow, and the Stage 0
route. The executable Stage 0 screen is `02_config_screening_colab.ipynb`.


## Colab Opening Workflow

1. Upload or open `notebooks/02_config_screening_colab.ipynb` in Google Colab.
2. Run setup cells in order. The notebook authenticates Google Drive API access
   when a Stage 0 run switch is enabled.
3. Do not mount MyDrive for Stage 0 data. The raw ticker files are
   downloaded by file ID into `/content/stage0_raw_stock_data`.
4. Outputs are written to `/content/stage0_config_screening_results`. Download
   or copy them after the run finishes.
5. Keep `RUN_STAGE0S`, `RUN_STAGE0A1`, `RUN_STAGE0A2`, and `RUN_STAGE0B` false
   until choosing the specific stage to run.


In [6]:
from pathlib import Path
import pandas as pd

RESULT_SCOPE = "validation_only"
TICKERS = ("CSCO", "JPM", "KO", "MSFT", "WMT")
RAW_DRIVE_FOLDER_ID = "154SlcH3nViUcvPXFBM-E4NPg_ybljBTG"
RAW_DRIVE_FOLDER_NAME = "s&p 100 adjusted 1 min data"

RAW_DRIVE_FILES = {
    "CSCO": {"name": "CSCO.txt", "file_id": "17A49kUiMELuQqdkOhw1KrpudjP5i5xIN"},
    "JPM": {"name": "JPM.txt", "file_id": "11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q"},
    "KO": {"name": "KO.txt", "file_id": "1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn"},
    "MSFT": {"name": "MSFT.txt", "file_id": "1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs"},
    "WMT": {"name": "WMT.txt", "file_id": "1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c"},
}

raw_data_manifest = pd.DataFrame([
    {
        "ticker": ticker,
        "file_name": item["name"],
        "drive_file_id": item["file_id"],
        "drive_folder_id": RAW_DRIVE_FOLDER_ID,
    }
    for ticker, item in RAW_DRIVE_FILES.items()
])
raw_data_manifest


,ticker,file_name,drive_file_id,drive_folder_id
0,CSCO,CSCO.txt,17A49kUiMELuQqdkOhw1KrpudjP5i5xIN,154SlcH3nViUcvPXFBM-E4NPg_ybljBTG
1,JPM,JPM.txt,11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q,154SlcH3nViUcvPXFBM-E4NPg_ybljBTG
2,KO,KO.txt,1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn,154SlcH3nViUcvPXFBM-E4NPg_ybljBTG
3,MSFT,MSFT.txt,1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs,154SlcH3nViUcvPXFBM-E4NPg_ybljBTG
4,WMT,WMT.txt,1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c,154SlcH3nViUcvPXFBM-E4NPg_ybljBTG


## Stage 0 Configuration Screen

Stage 0 is a fresh validation-only configuration screen over
`label_config + feature_set + window_size`. It starts from the raw ticker files
and the active freeze `docs/CONFIG_SCREENING_FREEZE_2026-06-04.md`.

Locked layers:

```text
Stage 0S  runtime/schema smoke, no selection
Stage 0A1 label-feature screen with LogReg + LightGBM at window_size=10
Stage 0A2 LightGBM window sensitivity on Stage 0A1 short list
Stage 0B  LogReg + LightGBM + simple GRU + MS-DLinear+TCN second-view
```

Important naming note: the table below is a pre-run design grid, not a results
table. Label, feature, window, candidate, mean, and LCB names are protocol
identifiers or allowed decision roles. They do not claim that a configuration is
already good before validation.

The stratified dummy is a lower-bound comparator only. It exists to report
`delta_macro_f1_vs_dummy`; it is not a model configuration candidate.


In [7]:
stage0_grid = pd.DataFrame([
    {
        "stage": "Stage 0S",
        "models": "LightGBM",
        "label_configs": "h09_bps3p0",
        "feature_sets": "price_volume_time",
        "window_sizes": "10",
        "seeds": "101",
        "selection_role": "schema/runtime only",
    },
    {
        "stage": "Stage 0A1",
        "models": "LogReg, LightGBM",
        "label_configs": "h03_bps1p5, h09_bps3p0, h24_bps7p5",
        "feature_sets": "price_action_core, technical_price, price_volume_time",
        "window_sizes": "10",
        "seeds": "101,202,303,404,505",
        "selection_role": "label-feature short list",
    },
    {
        "stage": "Stage 0A2",
        "models": "LightGBM",
        "label_configs": "Stage 0A1 short list",
        "feature_sets": "Stage 0A1 short list",
        "window_sizes": "5, 10, 20",
        "seeds": "101,202,303,404,505",
        "selection_role": "final config candidates",
    },
    {
        "stage": "Stage 0B",
        "models": "LogReg, LightGBM, simple GRU, MS-DLinear+TCN",
        "label_configs": "Stage 0A2 candidates",
        "feature_sets": "Stage 0A2 candidates",
        "window_sizes": "Stage 0A2 candidates",
        "seeds": "101,202,303,404,505",
        "selection_role": "second-view diagnostic only",
    },
])
stage0_grid


,stage,models,label_configs,feature_sets,window_sizes,seeds,selection_role
0,Stage 0S,LightGBM,h09_bps3p0,price_volume_time,10,101,schema/runtime only
1,Stage 0A1,"LogReg, LightGBM","h03_bps1p5, h09_bps3p0, h24_bps7p5","price_action_core, technical_price, price_volu...",10,"101,202,303,404,505",label-feature short list
2,Stage 0A2,LightGBM,Stage 0A1 short list,Stage 0A1 short list,"5, 10, 20","101,202,303,404,505",final config candidates
3,Stage 0B,"LogReg, LightGBM, simple GRU, MS-DLinear+TCN",Stage 0A2 candidates,Stage 0A2 candidates,Stage 0A2 candidates,"101,202,303,404,505",second-view diagnostic only


## Safety Boundary

- No holdout/test rows are loaded, transformed, windowed, scored, summarized,
  or used for wording decisions.
- Train and validation boundaries are chronological.
- Scaling fits train rows only.
- Input windows and label horizons do not cross tickers, splits, or trading
  days.
- Stage 0B cannot rerank or retract Stage 0A candidates. It may only set
  `deep_model_disagrees=True`.
- If zero Stage 0A cells pass `basic_gate`, the result is
  `do_not_decide_config`; do not run Stage 0B and do not start model-family
  screening.
